# Propagação de incertezas

Projeto visando a propagação de incertezas usando a biblioteca Uncertanties.

O experimento será uma simples titulação de HCl usando NaOH.

In [18]:
from uncertainties import ufloat
from uncertainties.umath import *

import numpy as np
import matplotlib.pyplot as plt

import scipy.stats as ss

In [48]:
#Dados a serem usados

volumes_NaOH_lista = [
    22.45, 22.38, 22.42, 22.50, 22.40, 
    22.35, 22.48, 22.44, 22.39, 22.52,
    22.41, 22.46, 22.37, 22.43, 22.49,
    22.40, 22.44, 22.36, 22.55, 22.42
]

#incertezas instrumentais (Tipo B): levar em conta distribuição retangular
bureta_50_mL = 0.05 / sqrt(3)
aliquota_hcl_25_mL = 0.03 / sqrt(3)

#incerteza NaOH
con_NaOH = 0.1025
inc_con_NaOH = 0.0005

In [49]:
#1) Analisando os volumes de NaOH

media_volume = np.mean(volumes_NaOH_lista)
desvio_padrao_volume = np.std(volumes_NaOH_lista)

print(f'Media do volume de NaOH: {media_volume} mL')
print(f'Desvio padrão do volume de NaOH: {round(desvio_padrao_volume, 4)}\n')

#Vendo a distribuição dos erros: 
erros_volume = volumes_NaOH_lista - media_volume

s, p = ss.shapiro(erros_volume)

print(f'Valor p (Shapiro): {round(p, 4)}: {'Segue normalidade' if p > 0.05 else 'Não segue normalidade'}')

Media do volume de NaOH: 22.433 mL
Desvio padrão do volume de NaOH: 0.053

Valor p (Shapiro): 0.7843: Segue normalidade


In [76]:
#ajustando as incertezas

incerteza_volume_NaOH_TipoA = desvio_padrao_volume/sqrt(len(volumes_NaOH_lista))
print(f'Incerteza do Tipo A - Volume de NaOH: +/-{round(incerteza_volume_NaOH_TipoA,4)}')

incerteza_combinada = sqrt(incerteza_volume_NaOH_TipoA**2 + bureta_50_mL**2)
print(f"Incerteza combinada (A + B): +/-{round(incerteza_combinada, 4)}")

Incerteza do Tipo A - Volume de NaOH: +/-0.0119
Incerteza combinada (A + B): +-0.0312


In [51]:
#Propagando as incertezas

Vnaoh = ufloat(media_volume, incerteza_combinada) #Primeira incerteza do volume
Cnaoh = ufloat(con_NaOH, inc_con_NaOH) #Incerteza da concentração
Vhcl = ufloat(25.00, aliquota_hcl_25_mL)

#As incertezas são relativas

Chcl = Vnaoh * Cnaoh / Vhcl

print(f'Concentração HCl: {Chcl} mol/L')
print(f'Concentração nominal: {round(Chcl.nominal_value,4)} mol/L')
print(f'Incerteza (Desvio P.): {round(Chcl.std_dev,4)} mol/L')

Concentração HCl: 0.0920+/-0.0005 mol/L
Concentração nominal: 0.092 mol/L
Incerteza (Desvio P.): 0.0005 mol/L


In [70]:
#Relatório final: incertezas de 68% para 95%

incerteza_95 = 2 * Chcl.std_dev

print(f'Concentração final 95% de confiança: {round(Chcl.nominal_value, 4)} +/- {round(incerteza_95, 4)} mol/L\n')

#Error_Components --> Mostra quem teve maior contribuição para o erro.

componentes = Chcl.error_components()
print('Componentes de erro:')
for variavel, erro in componentes.items():
    
    var_individual = (erro**2) / (Chcl.std_dev**2) * 100
    
    print(f'Variável: {variavel} | Incerteza (Contribuição): {round(erro,4)} | Em porcentagem: {round(var_individual,4)}')

Concentração final 95% de confiança: 0.092 +/- 0.0009 mol/L

Componentes de erro:
Variável: 25.000+/-0.017 | Incerteza (Contribuição): 0.0001 | Em porcentagem: 1.8313
Variável: 0.1025+/-0.0005 | Incerteza (Contribuição): 0.0004 | Em porcentagem: 90.7853
Variável: 22.433+/-0.031 | Incerteza (Contribuição): 0.0001 | Em porcentagem: 7.3834


In [75]:
#Vendo as derivadas com relação as variáveis.

derivadas = Chcl.derivatives

for variavel, derivada in derivadas.items():
    print(f"Derivada em relação a {variavel}: {derivada:.4f}")

Derivada em relação a 25.000+/-0.017: -0.0037
Derivada em relação a 0.1025+/-0.0005: 0.8973
Derivada em relação a 22.433+/-0.031: 0.0041
